# Assignment (Modul 4) - Lambda, Currying, Closure & Decorator
## Sistem Pencatatan Data Eksperimen Kecepatan Internet
**Nama:** Farid Al Farizi
**NIM:** 202410370110017

Melanjutkan fitur **Analisis Kualitas Koneksi** dari Modul 3. Pada modul ini program dikembangkan dengan **lambda expression**, **currying**, **closure**, dan **decorator**.

## 1. Dataset (lanjutan dari Modul 3)

In [1]:
# (NIM: 202410370110017)

from functools import reduce, wraps
import copy
import time

_t = lambda i, owner, loc, conn, dl, ul, ping, stable: {
    "test_id": i, "owner": owner, "location": loc, "connection": conn,
    "download_mbps": dl, "upload_mbps": ul, "ping_ms": ping, "stable": stable}

SPEED_TESTS = [
    _t(1,  "farid17", "Lab Informatika UMM",   "WiFi",    48.5, 20.3, 14, True),
    _t(2,  "nayla02", "Kos Tlogomas",          "WiFi",    22.7,  8.1, 31, False),
    _t(3,  "dika05",  "Alun-Alun Batu",        "Seluler", 15.2,  5.6, 45, True),
    _t(4,  "farid17", "Perpustakaan UMM",      "WiFi",    35.9, 15.2, 18, True),
    _t(5,  "farid17", "Rumah Jember",          "LAN",     95.4, 40.1,  9, True),
    _t(6,  "rani08",  "Cafe Dinoyo",           "WiFi",    12.3,  4.4, 52, False),
    _t(7,  "bima11",  "Stasiun Malang",        "Seluler", 28.6, 10.9, 38, True),
    _t(8,  "salsa03", "GOR UMM",               "Seluler",  9.8,  3.1, 67, False),
    _t(9,  "yoga21",  "Kos Landungsari",       "WiFi",    41.0, 18.7, 21, True),
    _t(10, "intan09", "Lab Jaringan UMM",      "LAN",     88.2, 37.5, 11, True),
    _t(11, "farid17", "Desa Petungsewu",       "Seluler",  6.4,  2.0, 89, False),
    _t(12, "hafiz14", "Masjid AR Fachruddin",  "WiFi",    30.5, 12.0, 25, True),
    _t(13, "citra06", "Matos",                 "Seluler", 19.7,  7.3, 40, True),
    _t(14, "rizky19", "Terminal Arjosari",     "Seluler", 11.1,  4.9, 58, False),
    _t(15, "laras12", "Kos Tegalgondo",        "WiFi",    26.4,  9.6, 29, True),
]

ORIGINAL = copy.deepcopy(SPEED_TESTS)
print(len(SPEED_TESTS), "catatan eksperimen")

15 catatan eksperimen


## 2. Lambda Expression
Fungsi `def` dari Modul 3 yang isinya hanya **satu expression** diganti menjadi lambda:

| Modul 3 (`def`) | Modul 4 (lambda) |
|---|---|
| `def is_stable(test): return test["stable"]` | `is_stable = lambda test: test["stable"]` |
| `def get_download(test): return test["download_mbps"]` | `get_download = lambda test: test["download_mbps"]` |
| `def add(a, b): return a + b` | `add = lambda a, b: a + b` |
| `classify_speed` (if bertingkat) | lambda dengan conditional expression |

In [2]:
# (NIM: 202410370110017)

is_stable      = lambda test: test["stable"]
get_download   = lambda test: test["download_mbps"]
add            = lambda a, b: a + b
classify_speed = lambda dl: "Cepat" if dl >= 40 else ("Sedang" if dl >= 15 else "Lambat")

# HOF dari Modul 3 yang isinya cuma satu expression juga bisa jadi lambda
apply_label = lambda classifier: lambda test: {**test, "quality": classifier(test["download_mbps"])}
label_quality = apply_label(classify_speed)

print(classify_speed(48.5), classify_speed(22.7), classify_speed(6.4))
print(add(2, 3), get_download(SPEED_TESTS[0]), is_stable(SPEED_TESTS[1]))
print(label_quality(SPEED_TESTS[0])["quality"])

Cepat Sedang Lambat
5 48.5 False
Cepat


## 3. Currying & Closure
Pada Modul 3, `make_filter(field, minimum)` menerima **dua parameter sekaligus**. Sekarang fungsi predikat dibuat **ter-curry**: `match(field)(op)(value)(test)` — setiap fungsi hanya menerima **satu parameter**.

Urutan parameter:
1. `field` diberikan **paling awal** (kolom yang dicek, misal `"download_mbps"`), lalu diingat oleh closure.
2. `op` (cara membandingkan: `>=`, `<=`, `==`).
3. `value` (nilai batas).
4. `test` (catatan data) diberikan **terakhir** karena berubah-ubah saat dipakai di `filter()`.

In [3]:
# (NIM: 202410370110017)

# Versi lambda (curried)
match = lambda field: lambda op: lambda value: lambda test: op(test[field], value)

# Operator perbandingan sebagai lambda
at_least = lambda a, b: a >= b
at_most  = lambda a, b: a <= b
equals   = lambda a, b: a == b

# Versi nested function (curried) agar closure terlihat jelas
def match_nested(field):
    def with_op(op):
        def with_value(value):
            def check(test):
                return op(test[field], value)   # field, op, value diambil dari enclosing scope
            return check
        return with_value
    return with_op

# Pemberian parameter bertahap -> fungsi-fungsi baru yang bisa dipakai ulang
by_download   = match("download_mbps")          # field "dikunci"
download_min  = by_download(at_least)            # field + op "dikunci"
is_fast       = download_min(40)                 # predikat siap pakai
is_medium_up  = download_min(15)

is_owned_by   = match("owner")(equals)           # pengganti is_owned_by Modul 3
is_quality    = match("quality")(equals)
low_ping      = match_nested("ping_ms")(at_most)(20)

print("Tes cepat (>=40)   :", [t["test_id"] for t in filter(is_fast, SPEED_TESTS)])
print("Tes >=15 Mbps      :", [t["test_id"] for t in filter(is_medium_up, SPEED_TESTS)])
print("Ping <= 20 ms      :", [t["test_id"] for t in filter(low_ping, SPEED_TESTS)])
print("Milik farid17      :", [t["test_id"] for t in filter(is_owned_by("farid17"), SPEED_TESTS)])

# Bukti closure: nilai dari enclosing function tersimpan di __closure__
print("\nIsi closure low_ping:", [c.cell_contents for c in low_ping.__closure__ if not callable(c.cell_contents)])

Tes cepat (>=40)   : [1, 5, 9, 10]
Tes >=15 Mbps      : [1, 2, 3, 4, 5, 7, 9, 10, 12, 13, 15]
Ping <= 20 ms      : [1, 4, 5, 10]
Milik farid17      : [1, 4, 5, 11]

Isi closure low_ping: ['ping_ms', 20]


## 4. Decorator
Tiga decorator ditambahkan tanpa mengubah isi fungsi aslinya:
- `@log_call` → mencatat nama fungsi, argumen, dan waktu eksekusi.
- `@validate_tests` → memastikan argumen `tests` berupa list berisi dict yang punya key wajib; kalau tidak valid, fungsi asli tidak dijalankan.
- `@memoize_by_user` → menyimpan hasil laporan per username (cache) menggunakan closure, sehingga pemanggilan ulang tidak menghitung ulang.

In [4]:
# (NIM: 202410370110017)

def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = (time.perf_counter() - start) * 1000
        extra = [a for a in args if not isinstance(a, list)]
        print(f"[LOG] {func.__name__}({', '.join(map(repr, extra))}) selesai dalam {elapsed:.3f} ms")
        return result
    return wrapper


REQUIRED_KEYS = {"owner", "location", "download_mbps", "stable"}

def validate_tests(func):
    @wraps(func)
    def wrapper(tests, *args, **kwargs):
        if not isinstance(tests, list) or not all(isinstance(t, dict) and REQUIRED_KEYS <= t.keys() for t in tests):
            return f"[ERROR] {func.__name__}: data tidak valid."
        return func(tests, *args, **kwargs)
    return wrapper


def memoize_by_user(func):
    cache = {}                                   # closure: cache bertahan antar pemanggilan
    @wraps(func)
    def wrapper(tests, username):
        if username in cache:
            print(f"[CACHE] hasil {username} diambil dari cache")
            return cache[username]
        cache[username] = func(tests, username)
        return cache[username]
    return wrapper

## 5. Recursion (dari Modul 3) + Decorator pada Fitur Laporan

In [5]:
# (NIM: 202410370110017)

def find_fastest(tests):
    if len(tests) == 1:
        return tests[0]
    best = find_fastest(tests[1:])
    return tests[0] if get_download(tests[0]) >= get_download(best) else best


count_if = lambda tests, predicate: 0 if not tests else (1 if predicate(tests[0]) else 0) + count_if(tests[1:], predicate)


@log_call
@validate_tests
@memoize_by_user
def quality_report(tests, username):
    mine = list(filter(is_owned_by(username), tests))
    if not mine:
        return f"--- Analisis {username} ---\n  Belum ada data."
    labeled = list(map(label_quality, mine))
    total = reduce(add, map(get_download, mine), 0)
    fastest = find_fastest(mine)
    q = lambda name: count_if(labeled, is_quality(name))
    return "\n".join([
        f"--- Analisis Kualitas Koneksi {username} ---",
        f"  Jumlah tes        : {len(mine)}",
        f"  Rata-rata download: {total / len(mine):.2f} Mbps",
        f"  Cepat / Sedang / Lambat: {q('Cepat')} / {q('Sedang')} / {q('Lambat')}",
        f"  Tes stabil        : {count_if(mine, is_stable)}",
        f"  Lokasi tercepat   : {fastest['location']} ({fastest['download_mbps']} Mbps)",
    ])


print(quality_report(SPEED_TESTS, "farid17"), end="\n\n")
print(quality_report(SPEED_TESTS, "farid17"), end="\n\n")   # kedua kali -> dari cache
print(quality_report(SPEED_TESTS, "nayla02"), end="\n\n")
print(quality_report(SPEED_TESTS, "tidak_ada"), end="\n\n")
print(quality_report("bukan list", "farid17"))

[LOG] quality_report('farid17') selesai dalam 0.055 ms
--- Analisis Kualitas Koneksi farid17 ---
  Jumlah tes        : 4
  Rata-rata download: 46.55 Mbps
  Cepat / Sedang / Lambat: 2 / 1 / 1
  Tes stabil        : 3
  Lokasi tercepat   : Rumah Jember (95.4 Mbps)

[CACHE] hasil farid17 diambil dari cache
[LOG] quality_report('farid17') selesai dalam 0.015 ms
--- Analisis Kualitas Koneksi farid17 ---
  Jumlah tes        : 4
  Rata-rata download: 46.55 Mbps
  Cepat / Sedang / Lambat: 2 / 1 / 1
  Tes stabil        : 3
  Lokasi tercepat   : Rumah Jember (95.4 Mbps)

[LOG] quality_report('nayla02') selesai dalam 0.021 ms
--- Analisis Kualitas Koneksi nayla02 ---
  Jumlah tes        : 1
  Rata-rata download: 22.70 Mbps
  Cepat / Sedang / Lambat: 0 / 1 / 0
  Tes stabil        : 0
  Lokasi tercepat   : Kos Tlogomas (22.7 Mbps)

[LOG] quality_report('tidak_ada') selesai dalam 0.007 ms
--- Analisis tidak_ada ---
  Belum ada data.

[LOG] quality_report('bukan list', 'farid17') selesai dalam 0.001 m

## 6. Bukti Fungsi Asli Tidak Berubah & Immutability

In [6]:
# (NIM: 202410370110017)

original_report = quality_report.__wrapped__.__wrapped__.__wrapped__   # fungsi asli tanpa decorator
print("Nama fungsi tetap   :", quality_report.__name__)
print("Panggil fungsi asli :")
print(original_report(SPEED_TESTS, "farid17"))
print("\nSPEED_TESTS tidak berubah:", SPEED_TESTS == ORIGINAL)
print("Ada key 'quality' di data asli?", "quality" in SPEED_TESTS[0])

Nama fungsi tetap   : quality_report
Panggil fungsi asli :
--- Analisis Kualitas Koneksi farid17 ---
  Jumlah tes        : 4
  Rata-rata download: 46.55 Mbps
  Cepat / Sedang / Lambat: 2 / 1 / 1
  Tes stabil        : 3
  Lokasi tercepat   : Rumah Jember (95.4 Mbps)

SPEED_TESTS tidak berubah: True
Ada key 'quality' di data asli? False


## 7. Penjelasan Decorator

| Decorator | Diberikan pada | Perilaku tambahan (tanpa mengubah fungsi asli) |
|---|---|---|
| `@log_call` | `quality_report` | Mencetak log nama fungsi, argumen (username), dan lama eksekusi dalam ms. |
| `@validate_tests` | `quality_report` | Mengecek data input sebelum fungsi dijalankan; jika bukan list of dict dengan key wajib, mengembalikan pesan error dan fungsi asli tidak dipanggil. |
| `@memoize_by_user` | `quality_report` | Menyimpan hasil laporan per username di `cache` (closure). Pemanggilan kedua untuk user yang sama langsung diambil dari cache. |

Isi `quality_report` sama sekali tidak berisi kode logging, validasi, atau cache — semua ditambahkan dari luar. Fungsi aslinya tetap bisa diakses melalui `__wrapped__` (berkat `functools.wraps`) dan menghasilkan laporan yang sama (dibuktikan di bagian 6).

## 8. Alasan Penggunaan Konsep

| Konsep | Dipakai di | Alasan |
|---|---|---|
| **Lambda Expression** | `is_stable`, `get_download`, `add`, `classify_speed`, `apply_label`, `count_if`, operator `at_least`/`at_most`/`equals` | Fungsi-fungsi ini hanya berisi satu expression, jadi lambda membuat kode lebih ringkas tanpa kehilangan arti. Cocok untuk operasi sederhana yang dikirim ke `map()`, `filter()`, dan `reduce()`. |
| **Currying** | `match(field)(op)(value)(test)` dan `match_nested` | Parameter diberikan bertahap sehingga dari satu fungsi umum bisa dibuat banyak predikat khusus (`is_fast`, `is_owned_by`, `is_quality`, `low_ping`). `field` diberikan pertama karena paling jarang berubah, `test` terakhir karena diisi oleh `filter()`. |
| **Closure** | Fungsi dalam `match`/`match_nested`, `apply_label`, cache di `memoize_by_user` | Fungsi dalam tetap mengingat nilai dari enclosing function (`field`, `op`, `value`, `classifier`, `cache`) walaupun fungsi luar sudah selesai dijalankan. Dibuktikan lewat `__closure__`. |
| **Decorator** | `@log_call`, `@validate_tests`, `@memoize_by_user` pada `quality_report` | Menambahkan logging, validasi, dan caching tanpa menyentuh logika laporan. Perilaku tambahan bisa dipasang/dilepas dan dipakai ulang di fungsi lain (separation of concerns). |

Semua transformasi data tetap **pure**: `apply_label` membuat dict baru, sehingga `SPEED_TESTS` tidak berubah.